In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

# Repo: https://github.com/zhaoziheng/SAT
# Checkpoints: https://huggingface.co/zzh99/SAT

#Install + clone
!pip -q install -U pip
!git clone -q https://github.com/zhaoziheng/SAT.git

%cd /content/SAT

# Install repo deps
!pip -q install -r requirements.txt

%cd /content/SAT/model
!pip -q install -e dynamic-network-architectures-main
%cd /content/SAT


import os, json, glob, pathlib
print("Working dir:", os.getcwd())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.3 MB/s eta 0:00:00
/content/SAT
/content/SAT/model
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for dynamic_network_architectures (pyproject.toml) ... done
/content/SAT
Working dir: /content/SAT


In [3]:
#Paths + model variant selection
from huggingface_hub import hf_hub_download
import os

CKPT_DIR = "/content/checkpoints_sat"
OUT_DIR  = "/content/sat_results"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)


#   "SAT-Pro"   (UNET-L backbone)
#   "SAT-Nano"  (UNET backbone, 72-dataset version)

SAT_VARIANT = "SAT-Nano"

# Map variant -> (vision_backbone, checkpoint file, text_encoder_checkpoint file)
VARIANT_MAP = {
    "SAT-Pro":   ("UNET-L", "Pro/SAT_Pro.pth",                "Pro/text_encoder.pth"),
    "SAT-Nano":  ("UNET",   "Nano/nano.pth",                 "Nano/nano_text_encoder.pth"),
    "UNET-Ours": ("UNET",   "Others/UNET-Ours/nano.pth",     "Others/UNET-Ours/text_encoder.pth"),
    "UNET-CPT":  ("UNET",   "Others/UNET-CPT/nano.pth",      "Others/UNET-CPT/text_encoder.pth"),
    "UNET-BB":   ("UNET",   "Others/UNET-BaseBERT/nano.pth", "Others/UNET-BaseBERT/text_encoder.pth"),
    "UMamba-CPT":("UMamba", "Others/UMamba-CPT/nano.pth",    "Others/UMamba-CPT/text_encoder.pth"),
    "SwinUNETR-CPT":("SwinUNETR","Others/SwinUNETR-CPT/nano.pth","Others/SwinUNETR-CPT/text_encoder.pth"),
}

vision_backbone, ckpt_rel, txt_ckpt_rel = VARIANT_MAP[SAT_VARIANT]

# Text encoder flag expected by SAT inference.py
TEXT_ENCODER = {
    "SAT-Pro": "ours",
    "SAT-Nano": "ours",
    "UNET-Ours": "ours",
    "UNET-CPT": "medcpt",
    "UMamba-CPT": "medcpt",
    "SwinUNETR-CPT": "medcpt",
    "UNET-BB": "basebert",
}[SAT_VARIANT]

sat_ckpt_path = hf_hub_download(
    repo_id="zzh99/SAT",
    filename=ckpt_rel,
    local_dir=CKPT_DIR,
    local_dir_use_symlinks=False,
)
txt_ckpt_path = hf_hub_download(
    repo_id="zzh99/SAT",
    filename=txt_ckpt_rel,
    local_dir=CKPT_DIR,
    local_dir_use_symlinks=False,
)

print("Variant:", SAT_VARIANT)
print("vision_backbone:", vision_backbone)
print("text_encoder:", TEXT_ENCODER)
print("SAT checkpoint:", sat_ckpt_path)
print("Text encoder checkpoint:", txt_ckpt_path)
print("Outputs:", OUT_DIR)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Nano/nano.pth:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Nano/nano_text_encoder.pth:   0%|          | 0.00/443M [00:00<?, ?B/s]

Variant: SAT-Nano
vision_backbone: UNET
text_encoder: ours
SAT checkpoint: /content/checkpoints_sat/Nano/nano.pth
Text encoder checkpoint: /content/checkpoints_sat/Nano/nano_text_encoder.pth
Outputs: /content/sat_results


In [4]:
# Build jsonl for inference

# jsonl fields per sample:
#   image: path to NIfTI, shape H,W,D
#   label: list[str] of targets (supported labels per paper)
#   dataset: string used to organize outputs
#   modality: "ct" | "mri" | "pet"
#   orientation_code: "RAS" (default, axial) or "ASR" (often for sagittal)

import json, glob, os

INPUT_NIFTI_DIR = "/content/drive/MyDrive/nnunet_inputs/imagesTs"
DATASET_NAME = "LGG"
MODALITY = "mri"
ORIENTATION_CODE = "RAS"

LABELS = [
    "Enhancing tumor core",   # <-- change to supported targets
]

JSONL_PATH = "/content/sat_infer.jsonl"

vol_paths = sorted(glob.glob(os.path.join(INPUT_NIFTI_DIR, "*.nii*")))
assert len(vol_paths) > 0, f"No NIfTI files found in: {INPUT_NIFTI_DIR}"

with open(JSONL_PATH, "w") as f:
    for p in vol_paths:
        rec = {
            "image": os.path.abspath(p),
            "label": LABELS,
            "dataset": DATASET_NAME,
            "modality": MODALITY,
            "orientation_code": ORIENTATION_CODE,
        }
        f.write(json.dumps(rec) + "\n")

print(f"Wrote {len(vol_paths)} records -> {JSONL_PATH}")
print("First record:")
print(rec)


Wrote 110 records -> /content/sat_infer.jsonl
First record:
{'image': '/content/drive/MyDrive/nnunet_inputs/imagesTs/TCGA_HT_A61B_0000.nii.gz', 'label': ['Enhancing tumor core'], 'dataset': 'LGG', 'modality': 'mri', 'orientation_code': 'RAS'}


In [5]:
%%bash
cd /content/SAT

python - << 'PY'
import re, pathlib

p = pathlib.Path("model/build_model.py")
s = p.read_text()

if "weights_only" not in s:
    s = s.replace(
        "torch.load(checkpoint, map_location=device)",
        "torch.load(checkpoint, map_location=device, weights_only=False)"
    )
    p.write_text(s)

PY



In [6]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


In [7]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [8]:
#Run inference

import os

MAX_QUERIES = max(16, len(LABELS) + 32)
BATCHSIZE_3D = 1
MASTER_PORT = "1234"

cmd = f"""torchrun --nproc_per_node=1 --master_port {MASTER_PORT} inference.py \
  --rcd_dir '{OUT_DIR}' \
  --datasets_jsonl '{JSONL_PATH}' \
  --vision_backbone '{vision_backbone}' \
  --checkpoint '{sat_ckpt_path}' \
  --text_encoder '{TEXT_ENCODER}' \
  --text_encoder_checkpoint '{txt_ckpt_path}' \
  --max_queries {MAX_QUERIES} \
  --batchsize_3d {BATCHSIZE_3D}
"""

print(cmd)
!{cmd}


torchrun --nproc_per_node=1 --master_port 1234 inference.py   --rcd_dir '/content/sat_results'   --datasets_jsonl '/content/sat_infer.jsonl'   --vision_backbone 'UNET'   --checkpoint '/content/checkpoints_sat/Nano/nano.pth'   --text_encoder 'ours'   --text_encoder_checkpoint '/content/checkpoints_sat/Nano/nano_text_encoder.pth'   --max_queries 33   --batchsize_3d 1

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-04-21 01:18:14.613787: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-21 01:18:14.633619: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting 

In [9]:
# Evaluate Output
import os, glob
import nibabel as nib
import numpy as np

ds_dir = os.path.join(OUT_DIR, DATASET_NAME)
print("Dataset output dir:", ds_dir)
if not os.path.exists(ds_dir):
    raise FileNotFoundError("No dataset output folder found. Check inference logs and DATASET_NAME.")

all_nii = sorted(glob.glob(os.path.join(ds_dir, "**", "*.nii*"), recursive=True))
print("Total nifti outputs found:", len(all_nii))
print("First few:")
for p in all_nii[:10]:
    print(" -", p)

if all_nii:
    img = nib.load(all_nii[0])
    arr = img.get_fdata()
    print("\nLoaded:", all_nii[0])
    print("Shape:", arr.shape)
    print("dtype:", arr.dtype)
    print("min/max:", float(np.nanmin(arr)), float(np.nanmax(arr)))


Dataset output dir: /content/sat_results/LGG
Total nifti outputs found: 330
First few:
 - /content/sat_results/LGG/img_TCGA_CS_4941_0000.nii.gz
 - /content/sat_results/LGG/img_TCGA_CS_4942_0000.nii.gz
 - /content/sat_results/LGG/img_TCGA_CS_4943_0000.nii.gz
 - /content/sat_results/LGG/img_TCGA_CS_4944_0000.nii.gz
 - /content/sat_results/LGG/img_TCGA_CS_5393_0000.nii.gz
 - /content/sat_results/LGG/img_TCGA_CS_5395_0000.nii.gz
 - /content/sat_results/LGG/img_TCGA_CS_5396_0000.nii.gz
 - /content/sat_results/LGG/img_TCGA_CS_5397_0000.nii.gz
 - /content/sat_results/LGG/img_TCGA_CS_6186_0000.nii.gz
 - /content/sat_results/LGG/img_TCGA_CS_6188_0000.nii.gz

Loaded: /content/sat_results/LGG/img_TCGA_CS_4941_0000.nii.gz
Shape: (64, 128, 43)
dtype: float64
min/max: -1.320533275604248 3.4251739978790283


In [10]:
# ============================
# SAT EVALUATION (BioMedParse-style)
# ============================

import os, glob, re
import numpy as np
import pandas as pd
import nibabel as nib
from PIL import Image
from sklearn.metrics import jaccard_score, f1_score

# ---- REQUIRED: set these to match your run ----
SAT_OUT_DIR   = OUT_DIR         # e.g. "/content/sat_results'
SAT_DATASET   = DATASET_NAME    # e.g. "LGG" (must match what you wrote in jsonl)
DATASET_PATH  = "/content/dataset/lgg-mri-segmentation/kaggle_3m"  # kaggle_3m folder

# ---- OUTPUTS (match BioMedParse style) ----
EVAL_OUT_DIR = os.path.join(SAT_OUT_DIR, "eval", SAT_DATASET)
os.makedirs(EVAL_OUT_DIR, exist_ok=True)

print("SAT outputs root:", SAT_OUT_DIR)
print("SAT dataset name :", SAT_DATASET)
print("Kaggle dataset   :", DATASET_PATH)
print("Eval output dir  :", EVAL_OUT_DIR)

# -----------------------------
# Helpers (same logic as BioMedParse)
# -----------------------------
def list_sat_pred_niftis(sat_out_dir: str, sat_dataset: str):
    ds_dir = os.path.join(sat_out_dir, sat_dataset)
    all_nii = sorted(glob.glob(os.path.join(ds_dir, "**", "*.nii*"), recursive=True))
    if not all_nii:
        raise FileNotFoundError(
            f"No NIfTI outputs found under: {ds_dir}\n"
            f"Check SAT inference logs and that DATASET_NAME in jsonl == '{sat_dataset}'."
        )
    return all_nii

def normalize_patient_id(short_patient_id: str):
    # "TCGA_XX_YYYY" style from BioMedParse notebook
    return short_patient_id.strip()

def find_pred_for_patient(all_pred_paths, short_patient_id: str):
    """
    Robust match: find any predicted nifti path containing the patient id token.
    If multiple matches exist, prefer ones that look like final mask outputs.
    """
    pid = normalize_patient_id(short_patient_id)

    matches = [p for p in all_pred_paths if pid in os.path.basename(p) or pid in p]
    if not matches:
        # fallback: try looser match (some pipelines remove underscores)
        pid_loose = pid.replace("_", "")
        matches = [p for p in all_pred_paths if pid_loose in os.path.basename(p).replace("_","")]

    if not matches:
        return None

    # prefer files that look like segmentation/mask outputs
    def score(p):
        name = os.path.basename(p).lower()
        s = 0
        for key in ["mask", "seg", "pred", "prediction", "tumor"]:
            if key in name:
                s += 1
        # prefer .nii.gz
        if name.endswith(".nii.gz"):
            s += 1
        return s

    matches = sorted(matches, key=score, reverse=True)
    return matches[0]

def load_gt_mask_volume_from_tif(patient_dir: str):
    """
    Builds GT volume (D,H,W) from *_mask.tif slices, aligned by the non-mask image ordering.
    This matches your BioMedParse notebook pairing logic.
    """
    image_files = sorted([
        f for f in glob.glob(os.path.join(patient_dir, "*.tif"))
        if f[-5].isdigit() and "_mask" not in f
    ])
    if len(image_files) == 0:
        return None, None

    gt_slices = []
    paired = 0
    for img_file in image_files:
        base = os.path.splitext(img_file)[0]
        mask_file = f"{base}_mask.tif"
        if os.path.exists(mask_file):
            gt = np.array(Image.open(mask_file).convert("L"))
            gt = (gt > 0).astype(np.uint8)
            gt_slices.append(gt)
            paired += 1

    if paired == 0:
        return None, image_files

    gt_vol = np.stack(gt_slices, axis=0)  # (D,H,W)
    return gt_vol, image_files

def load_sat_pred_volume(pred_path: str):
    """
    Loads SAT prediction nifti and converts to (D,H,W) binary.
    Handles common axis orders by trying to interpret the depth axis.
    """
    img = nib.load(pred_path)
    arr = img.get_fdata()

    # Convert to numpy float -> binary later
    arr = np.asarray(arr)

    # Common cases:
    #  - (H,W,D) or (D,H,W)
    # We want (D,H,W). We'll guess by taking the smallest axis as depth is risky,
    # so we use heuristic: if one axis count is "slice-like" (<= 256) and others ~ 256/512.
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D volume, got shape {arr.shape} from {pred_path}")

    # Heuristic: if last axis looks like depth (often <= 256), assume (H,W,D)
    if arr.shape[2] <= 256 and arr.shape[0] >= 128 and arr.shape[1] >= 128:
        vol = np.transpose(arr, (2, 0, 1))  # (D,H,W)
    # If first axis looks like depth, assume already (D,H,W)
    elif arr.shape[0] <= 256 and arr.shape[1] >= 128 and arr.shape[2] >= 128:
        vol = arr
    else:
        # fallback: assume (H,W,D)
        vol = np.transpose(arr, (2, 0, 1))

    # binarize: SAT sometimes outputs probs/logits or 0/1
    # If values are not strictly 0/1, threshold at 0.5
    if vol.max() > 1 or vol.min() < 0:
        # logits-ish -> threshold at 0
        bin_vol = (vol > 0).astype(np.uint8)
    else:
        bin_vol = (vol > 0.5).astype(np.uint8)

    return bin_vol

def resize_2d_nearest(src_2d: np.ndarray, tgt_shape):
    """
    Nearest-neighbor resize without scipy (keeps this notebook light).
    """
    src_h, src_w = src_2d.shape
    tgt_h, tgt_w = tgt_shape
    if (src_h, src_w) == (tgt_h, tgt_w):
        return src_2d.astype(np.uint8)

    # map target indices -> source indices
    y_idx = (np.linspace(0, src_h - 1, tgt_h)).astype(np.int64)
    x_idx = (np.linspace(0, src_w - 1, tgt_w)).astype(np.int64)
    out = src_2d[np.ix_(y_idx, x_idx)]
    return out.astype(np.uint8)

def compute_patient_metrics(gt_vol: np.ndarray, pred_vol: np.ndarray):
    """
    Slice-wise IoU/F1, averaged across slices with any signal (same as BioMedParse notebook).
    """
    D = min(gt_vol.shape[0], pred_vol.shape[0])
    ious, f1s = [], []

    for z in range(D):
        gt = gt_vol[z].astype(np.uint8)
        pr = pred_vol[z].astype(np.uint8)

        if gt.shape != pr.shape:
            pr = resize_2d_nearest(pr, gt.shape)

        gt_sum = gt.sum()
        pr_sum = pr.sum()

        # Case 1: both empty → perfect score
        if gt_sum == 0 and pr_sum == 0:
            ious.append(1.0)
            f1s.append(1.0)

        # Case 2: one empty, one not → zero score
        elif gt_sum == 0 or pr_sum == 0:
            ious.append(0.0)
            f1s.append(0.0)

        # Case 3: both non-empty → compute normally
        else:
            ious.append(
                jaccard_score(gt.flatten(), pr.flatten(), zero_division=0)
            )
            f1s.append(
                f1_score(gt.flatten(), pr.flatten(), zero_division=0)
            )


    mean_iou = float(np.mean(ious)) if ious else 0.0
    mean_f1  = float(np.mean(f1s)) if f1s else 0.0
    return mean_iou, mean_f1, len(ious)

# -----------------------------
# Main loop (same structure as BioMedParse notebook)
# -----------------------------
data_csv = os.path.join(DATASET_PATH, "data.csv")
patient_info = pd.read_csv(data_csv)
patient_dirs = sorted(glob.glob(os.path.join(DATASET_PATH, "TCGA_*")))
print(f"Patients in data.csv: {len(patient_info)}")
print(f"Patient directories : {len(patient_dirs)}")

all_pred_paths = list_sat_pred_niftis(SAT_OUT_DIR, SAT_DATASET)
print(f"SAT NIfTI outputs found: {len(all_pred_paths)}")

results = []

for idx, patient_dir in enumerate(patient_dirs, 1):
    full_patient_id  = os.path.basename(patient_dir)
    short_patient_id = "_".join(full_patient_id.split("_")[:3])

    print(f"\n[{idx}/{len(patient_dirs)}] {short_patient_id}")

    # demographics
    row = patient_info[patient_info["Patient"] == short_patient_id]
    if len(row) == 0:
        print("  No demographics in data.csv, skipping")
        continue
    race   = row.iloc[0]["race"]
    gender = row.iloc[0]["gender"]
    print(f"  Race: {race}, Gender: {gender}")

    # GT
    gt_vol, image_files = load_gt_mask_volume_from_tif(patient_dir)
    if gt_vol is None:
        print("  No GT masks found, skipping")
        continue
    print(f"  GT volume shape: {gt_vol.shape}")

    # Pred
    pred_path = find_pred_for_patient(all_pred_paths, short_patient_id)
    if pred_path is None:
        print("  ⚠️  No SAT prediction found for this patient (name mismatch). Skipping.")
        continue

    print(f"  SAT pred path: {pred_path}")
    try:
        pred_vol = load_sat_pred_volume(pred_path)
    except Exception as e:
        print(f"  ⚠️  Failed to load pred volume: {str(e)[:200]}")
        continue

    print(f"  Pred volume shape: {pred_vol.shape}")

    # Metrics
    mean_iou, mean_f1, num_slices = compute_patient_metrics(gt_vol, pred_vol)
    print(f"  IoU: {mean_iou:.4f}, F1: {mean_f1:.4f} (slices used: {num_slices})")

    results.append({
        "Patient": short_patient_id,
        "Race": race,
        "Gender": gender,
        "IoU": mean_iou,
        "F1": mean_f1,
        "Num_Slices": num_slices,
        "Prediction_Path": pred_path,
    })

# -----------------------------
# Save results (exactly like BioMedParse notebook)
# -----------------------------
if len(results) == 0:
    print("\nNo results to save.")
else:
    results_df = pd.DataFrame(results)

    csv_path = os.path.join(EVAL_OUT_DIR, "sat_patient_metrics.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"\nSaved: {csv_path}")

    overall_iou = results_df["IoU"].mean()
    overall_f1  = results_df["F1"].mean()

    print(f"\n{'='*70}")
    print("OVERALL METRICS (SAT)")
    print(f"{'='*70}")
    print(f"  Mean IoU: {overall_iou:.4f}")
    print(f"  Mean F1:  {overall_f1:.4f}")
    print(f"  Patients: {len(results_df)}")

    race_stats = results_df.groupby("Race")[["IoU","F1"]].agg(["mean","std","count"]).round(4)
    print(f"\n{'='*70}")
    print("RACE-LEVEL METRICS (SAT)")
    print(f"{'='*70}")
    print(race_stats)
    race_csv = os.path.join(EVAL_OUT_DIR, "sat_race_metrics.csv")
    race_stats.to_csv(race_csv)

    gender_stats = results_df.groupby("Gender")[["IoU","F1"]].agg(["mean","std","count"]).round(4)
    print(f"\n{'='*70}")
    print("GENDER-LEVEL METRICS (SAT)")
    print(f"{'='*70}")
    print(gender_stats)
    gender_csv = os.path.join(EVAL_OUT_DIR, "sat_gender_metrics.csv")
    gender_stats.to_csv(gender_csv)

    summary_path = os.path.join(EVAL_OUT_DIR, "sat_summary.txt")
    with open(summary_path, "w") as f:
        f.write("="*70 + "\n")
        f.write("SAT SEGMENTATION RESULTS\n")
        f.write("="*70 + "\n\n")

        f.write("OVERALL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(f"Mean IoU: {overall_iou:.4f}\n")
        f.write(f"Mean F1:  {overall_f1:.4f}\n")
        f.write(f"Patients: {len(results_df)}\n\n")

        f.write("RACE-LEVEL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(race_stats.to_string())
        f.write("\n\n")

        f.write("GENDER-LEVEL METRICS\n")
        f.write("-"*70 + "\n")
        f.write(gender_stats.to_string())
        f.write("\n")

    print(f"\nSaved: {summary_path}")
    print("\n" + "="*70)
    print("🎉 SAT EVALUATION COMPLETE!")
    print("="*70)
    print(f"Results saved to: {EVAL_OUT_DIR}")
    print("Files created:")
    print("  - sat_patient_metrics.csv")
    print("  - sat_race_metrics.csv")
    print("  - sat_gender_metrics.csv")
    print("  - sat_summary.txt")


SAT outputs root: /content/sat_results
SAT dataset name : LGG
Kaggle dataset   : /content/dataset/lgg-mri-segmentation/kaggle_3m
Eval output dir  : /content/sat_results/eval/LGG


FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/lgg-mri-segmentation/kaggle_3m/data.csv'